# T2 орчуулгын туршилт — Colab (үнэгүй T4)

Вэбийн notebook-тэй ижил MyDrive/t2s/ файлууд. **Runtime → Run all.** D2 ба Qwen3.5-9B (4-bit) асаад dev (300), test (200)
асуултыг T2 горимоор орчуулна (маскыг DB-ийн нэрээр шүүх + нэр томьёоны RAG), I3t2-ийг ажиллуулна (~4–5 цаг).
Үр дүн split бүр дуусмагц **MyDrive/t2s/results_t2/**-д хуулагдана — Colab тасарсан ч дууссан хэсэг үлдэнэ.
Табыг нээлттэй, компьютерийг унтуулахгүй байлгана.

In [ ]:
import os, shutil, subprocess, zipfile
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
SRC = Path("/content/drive/MyDrive/t2s")
assert (SRC / "t2s_code.zip").exists(), "MyDrive/t2s/t2s_code.zip алга"
assert list(SRC.glob("t2s_adapter_d2.zip.part*")), "MyDrive/t2s/t2s_adapter_d2.zip.part00 … алга"
REPO = Path("/content/t2s")
shutil.rmtree(REPO, ignore_errors=True)
zipfile.ZipFile(SRC / "t2s_code.zip").extractall(REPO)
os.chdir(REPO)
os.environ.update(T2S_PG_PASSWORD="t2s_nb_admin", VLLM_USE_FLASHINFER_SAMPLER="0", ADAPTER_SRC=str(SRC),
                  STAGE="t2", RESULTS_DST=str(SRC / "results_t2"), HF_HUB_ENABLE_HF_TRANSFER="0")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)

def run(cmd):
    p = subprocess.Popen(["bash", "-c", f"source scripts/server/env.sh && {cmd}"], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")

In [ ]:
# PostgreSQL 16, vLLM, DB (seed 42) — ~10–15 мин
run("bash scripts/notebook/bootstrap.sh")

In [ ]:
# T2 орчуулга → I3t2 dev, test (~4–5 цаг)
run("bash scripts/server/run_plan_l4.sh")